# HbA1c — In-sample XAI extension

The reusable XAI framework is applied to the final logistic-regression interaction model fitted on the eligible full cohort.

The same cohort supplies both reference and explanation data. All explanations and ROC-AUC scoring are therefore in-sample and do not estimate out-of-sample predictive performance.

Bootstrap-refit stability is not assessed because the implementation provides no retraining callback. Results are exported to `../../results/hba1c`.



The exported global SHAP analysis uses SHAP 0.51.0 `PermutationExplainer`.


## Cohort and final-model preparation


In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "xai_extension":
    raise RuntimeError(
        "Run this notebook with the working directory set to "
        "thesis/src/xai_extension. Current directory: "
        f"{NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
FRAMEWORK_DIR = PROJECT_ROOT / "src" / "xai_framework"
if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from core.statsmodels_wrapper import StatsModelsWrapper
from run_pipeline import run_pipeline

data_path = "../../data/hba1c/diabetic_data_initial.csv"
map_path = "../../data/hba1c/id_mapping.csv"

df = pd.read_csv(data_path, keep_default_na=False)
idmap_raw = pd.read_csv(map_path, keep_default_na=False)



c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Parse the identifier mapping file.
idmap_raw.columns = ["id", "description"]
idmap_raw["id"] = idmap_raw["id"].astype(str).str.strip()
idmap_raw["description"] = idmap_raw["description"].astype(str).str.strip()


In [3]:
def parse_id_mapping(idmap_raw):
    """Parse the source identifier mapping into named lookup tables."""
    sections = {}
    current_section = None
    current_rows = []

    for _, row in idmap_raw.iterrows():
        left = str(row["id"]).strip()
        right = str(row["description"]).strip()

        if left in ["nan", "ID mappings for the diabetes dataset"]:
            continue
        if right in ["nan", "? - always means a missing value"]:
            continue

        if left in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
            if current_section is not None and current_rows:
                sections[current_section] = pd.DataFrame(current_rows, columns=["id", "description"])
            current_section = left
            current_rows = []
            continue

        if left == "description" or right == "description":
            continue

        try:
            key = int(float(left))
            current_rows.append((key, right))
        except Exception:
            pass

    if current_section is not None and current_rows:
        sections[current_section] = pd.DataFrame(current_rows, columns=["id", "description"])

    return sections

sections = parse_id_mapping(idmap_raw)


In [4]:
admission_type_map = dict(zip(sections["admission_type_id"]["id"], sections["admission_type_id"]["description"]))
discharge_map = dict(zip(sections["discharge_disposition_id"]["id"], sections["discharge_disposition_id"]["description"]))
admission_source_map = dict(zip(sections["admission_source_id"]["id"], sections["admission_source_id"]["description"]))


In [5]:
# Apply the source-study preprocessing.
df2 = df.copy()

# Define readmission within 30 days as the outcome.
df2["readmitted_bin"] = (df2["readmitted"] == "<30").astype(int)

# Remove variables excluded by the source study.
df2 = df2.drop(columns=["weight", "payer_code"], errors="ignore")

# Standardise the source missing-value markers.
df2["medical_specialty"] = df2["medical_specialty"].replace("?", np.nan).fillna("missing")
df2["race"] = df2["race"].replace("?", np.nan).fillna("Missing")

# Add readable descriptions for coded fields.
df2["discharge_desc"] = df2["discharge_disposition_id"].map(discharge_map)
df2["admission_type_desc"] = df2["admission_type_id"].map(admission_type_map)
df2["admission_source_desc"] = df2["admission_source_id"].map(admission_source_map)

# Retain the first encounter for each patient.
# Preserve the complete first encounter rather than combining rows.
df2 = (
    df2.sort_values(["patient_nbr", "encounter_id"])
       .drop_duplicates(subset="patient_nbr", keep="first")
       .copy()
)

# Exclude invalid or unknown gender records.
df2 = df2[df2["gender"].isin(["Female", "Male"])].copy()

# Exclude discharge dispositions associated with death or hospice.
death_hospice_ids = [11, 13, 14, 19, 20, 21]
df2 = df2[~df2["discharge_disposition_id"].isin(death_hospice_ids)].copy()

print("Shape after cohort selection:", df2.shape)


Shape after cohort selection: (69970, 52)


In [6]:
# Construct the derived variables.
def make_hba1c_group(row):
    """Assign the study-defined HbA1c and medication-change group."""
    a1c = str(row["A1Cresult"]).strip()
    change = str(row["change"]).strip()

    if a1c == "None":
        return "No test was performed"
    elif a1c in ["Norm", ">7"]:
        return "Normal result of the test"
    elif a1c == ">8" and change == "No":
        return "Result was high but the diabetic medication was not changed"
    elif a1c == ">8" and change == "Ch":
        return "Result was high and the diabetic medication was changed"
    else:
        return np.nan



df2["hba1c_group"] = df2.apply(make_hba1c_group, axis=1)


In [7]:
df2["discharge_group"] = np.where(
    df2["discharge_disposition_id"] == 1,
    "Discharged to home",
    "Otherwise"
)


In [8]:
def map_admission_source(x):
    """Map an admission-source identifier to the study category."""
    if x == 7:
        return "Admitted from emergency room"
    elif x in [1, 2, 3]:
        return "Admitted because of physician/clinic referral"
    else:
        return "Otherwise"

df2["admission_source_group"] = df2["admission_source_id"].apply(map_admission_source)


In [9]:
def map_specialty(x):
    """Collapse a medical specialty into the study categories."""
    if x == "InternalMedicine":
        return "Internal Medicine"
    elif x == "Cardiology":
        return "Cardiology"
    elif "Surgery" in str(x):
        return "Surgery"
    elif x in ["Family/GeneralPractice", "FamilyPractice"]:
        return "Family/general practice"
    elif x == "missing":
        return "Missing or unknown"
    else:
        return "Other"

df2["specialty_group"] = df2["medical_specialty"].apply(map_specialty)


In [10]:
def map_race(x):
    """Map race values to the study categories."""
    if x == "AfricanAmerican":
        return "African American"
    elif x == "Caucasian":
        return "Caucasian"
    elif x == "Missing":
        return "Missing"
    else:
        return "Other"

df2["race_group"] = df2["race"].apply(map_race)


In [11]:
def age_midpoint(age_str):
    """Convert an age interval to its numeric midpoint."""
    nums = [int(x) for x in str(age_str).replace("[", "").replace(")", "").split("-") if x.isdigit()]
    if len(nums) == 2:
        return (nums[0] + nums[1]) / 2
    return np.nan

df2["age_mid"] = df2["age"].apply(age_midpoint)

def map_age_group3(x):
    """Assign a numeric age to one of the three study groups."""
    if pd.isna(x):
        return np.nan
    elif x <= 30:
        return "30 years old or younger"
    elif x <= 60:
        return "30-60 years old"
    else:
        return "Older than 60"

df2["age_group3"] = df2["age_mid"].apply(map_age_group3)


In [12]:
def group_primary_diag(code):
    """Map a primary diagnosis code to the study category."""
    code = str(code).strip()

    if code.startswith("V") or code.startswith("E"):
        return "Other"
    if code.startswith("250"):
        return "Diabetes"

    m = re.match(r"^(\d+)", code)
    if not m:
        return "Other"

    num = int(m.group(1))

    if 390 <= num <= 459 or num == 785:
        return "Circulatory"
    elif 460 <= num <= 519 or num == 786:
        return "Respiratory"
    elif 520 <= num <= 579 or num == 787:
        return "Digestive"
    elif 800 <= num <= 999:
        return "Injury"
    elif 710 <= num <= 739:
        return "Musculoskeletal"
    elif 580 <= num <= 629 or num == 788:
        return "Genitourinary"
    elif 140 <= num <= 239:
        return "Neoplasms"
    else:
        return "Other"

df2["primary_diag_group"] = df2["diag_1"].apply(group_primary_diag)


In [13]:
# Construct the final analysis cohort.
df2["hba1c_group"] = df2.apply(make_hba1c_group, axis=1)

analysis_df = df2[[
    "readmitted_bin",
    "hba1c_group",
    "gender",
    "discharge_group",
    "admission_source_group",
    "specialty_group",
    "primary_diag_group",
    "race_group",
    "age_group3",
    "age_mid",
    "time_in_hospital"
]].dropna().copy()

print("analysis_df shape:", analysis_df.shape)
print(analysis_df["hba1c_group"].value_counts(dropna=False))


analysis_df shape: (69970, 11)
hba1c_group
No test was performed                                          57125
Normal result of the test                                       6606
Result was high and the diabetic medication was changed         4058
Result was high but the diabetic medication was not changed     2181
Name: count, dtype: int64


In [14]:
analysis_df["discharge_group"] = pd.Categorical(
    analysis_df["discharge_group"],
    categories=["Discharged to home", "Otherwise"]
)

analysis_df["race_group"] = pd.Categorical(
    analysis_df["race_group"],
    categories=["African American", "Caucasian", "Missing", "Other"]
)

analysis_df["admission_source_group"] = pd.Categorical(
    analysis_df["admission_source_group"],
    categories=[
        "Admitted from emergency room",
        "Admitted because of physician/clinic referral",
        "Otherwise"
    ]
)

analysis_df["specialty_group"] = pd.Categorical(
    analysis_df["specialty_group"],
    categories=[
        "Cardiology",
        "Family/general practice",
        "Internal Medicine",
        "Missing or unknown",
        "Other",
        "Surgery"
    ]
)

analysis_df["age_group3"] = pd.Categorical(
    analysis_df["age_group3"],
    categories=[
        "30-60 years old",
        "Older than 60",
        "30 years old or younger"
    ]
)

analysis_df["primary_diag_group"] = pd.Categorical(
    analysis_df["primary_diag_group"],
    categories=[
        "Diabetes",
        "Circulatory",
        "Digestive",
        "Genitourinary",
        "Injury",
        "Musculoskeletal",
        "Neoplasms",
        "Other",
        "Respiratory"
    ]
)



analysis_df["hba1c_group"] = pd.Categorical(
    analysis_df["hba1c_group"],
    categories=[
        "No test was performed",
        "Normal result of the test",
        "Result was high and the diabetic medication was changed",
        "Result was high but the diabetic medication was not changed"
    ],
    ordered=True
)


In [15]:
# Apply the categorical levels used by the fitted formula.
cat_cols = [
    "hba1c_group", "gender", "discharge_group", "admission_source_group",
    "specialty_group", "primary_diag_group", "race_group", "age_group3"
]

for c in cat_cols:
    analysis_df[c] = analysis_df[c].astype("category")


In [16]:
final_model = smf.logit(
    "readmitted_bin ~ gender + race_group + admission_source_group + "
    "discharge_group + specialty_group + primary_diag_group + "
    "age_group3 + time_in_hospital + hba1c_group + "
    "discharge_group:race_group + "
    "discharge_group:specialty_group + "
    "discharge_group:primary_diag_group + "
    "discharge_group:time_in_hospital + "
    "specialty_group:time_in_hospital + "
    "specialty_group:age_group3 + "
    "primary_diag_group:time_in_hospital + "
    "primary_diag_group:hba1c_group",
    data=analysis_df
).fit(maxiter=200)


Optimization terminated successfully.
         Current function value: 0.294692
         Iterations 8


In [17]:
# Build the aligned design matrix from the fitted Statsmodels specification.

X_design = pd.DataFrame(
    final_model.model.exog,
    columns=final_model.model.exog_names
)

y_design = pd.Series(
    final_model.model.endog,
    name="readmitted_bin"
)

feature_names = list(X_design.columns)

print("X_design shape:", X_design.shape)
print("y_design shape:", y_design.shape)
print("Number of features:", len(feature_names))


X_design shape: (69970, 91)
y_design shape: (69970,)
Number of features: 91


## Execute the XAI pipeline


In [18]:
# The framework names these fields X_train/X_test, but this paper has no
# held-out prediction design. Both refer to the eligible full-cohort design
# matrix, and all resulting explanations are labelled in-sample.
config = {
    "dataset_name": "hba1c",

    "model": final_model,

    "X_train": X_design,
    "X_test": X_design,

    "y_train": y_design,
    "y_test": y_design,

    "feature_names": feature_names,

    # Repeated local explanations
    "local_cases_per_type": 10,
    "lime_seeds": list(range(20)),
    "lime_num_features": 5,
    "lime_stability_top_k": 3,

    "scoring": "roc_auc",

    "output_dir": "../../results/hba1c"
}

experiment = run_pipeline(config)


PermutationExplainer explainer: 69971it [41:18, 28.24it/s]                             


Results exported to: ../../results/hba1c


## XAI framework output sanity checks — in-sample


In [19]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))


Dataset: hba1c
Model: BinaryResultsWrapper
X_train shape: (69970, 91)
X_test shape: (69970, 91)
y_train length: 69970
y_test length: 69970
Feature names: 91


### Input consistency


In [20]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("\nBasic shape validation: OK")



Basic shape validation: OK


### HbA1c design-matrix check


In [21]:
assert list(config["X_train"].columns) == config["feature_names"]
assert list(config["X_test"].columns) == config["feature_names"]

assert list(config["X_train"].columns) == list(final_model.model.exog_names)

print("Statsmodels design-matrix alignment: OK")


Statsmodels design-matrix alignment: OK


### Prediction-interface check


In [22]:
model = config["model"]

wrapped_model = StatsModelsWrapper(model)

probs = wrapped_model.predict_proba(
    config["X_test"]
)[:, 1]

preds = wrapped_model.predict(
    config["X_test"]
)

print("\nProbability range:", probs.min(), probs.max())
print(
    "Predictions:",
    np.unique(preds, return_counts=True)
)

assert np.all(probs >= 0)
assert np.all(probs <= 1)

print("Prediction test: OK")



Probability range: 0.009531647448095457 0.35282606409965933
Predictions: (array([0]), array([69970]))
Prediction test: OK


### Framework results


In [23]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print("\nGlobal result keys:")
print(global_results.keys())

print("\nLocal result keys:")
print(local_results.keys())

if stability_results is None:
    print("\nStability: correctly absent for HbA1c")
else:
    print("\nStability keys:")
    print(stability_results.keys())



Global result keys:
dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'shap_explainer_class', 'rank_compare', 'rank_agreement'])

Local result keys:
dict_keys(['cases', 'lime', 'lime_summary', 'lime_seed_stability'])

Stability: correctly absent for HbA1c


### Global explanations


In [24]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("\nNative importance:")
display(native_df.head(10))

print("\nPermutation importance:")
display(perm_df.head(10))

print("\nSHAP importance:")
display(shap_df.head(10))



Native importance:


,Feature,Importance,RawValue,Method
50,specialty_group[T.Surgery]:age_group3[T.30 yea...,2.989244,-2.989244,statsmodels
46,specialty_group[T.Family/general practice]:age...,2.498977,-2.498977,statsmodels
49,specialty_group[T.Other]:age_group3[T.30 years...,2.409895,-2.409895,statsmodels
47,specialty_group[T.Internal Medicine]:age_group...,2.023971,-2.023971,statsmodels
21,age_group3[T.30 years old or younger],1.857647,1.857647,statsmodels
48,specialty_group[T.Missing or unknown]:age_grou...,1.471311,-1.471311,statsmodels
71,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.802534,0.802534,statsmodels
63,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.796584,0.796584,statsmodels
16,primary_diag_group[T.Musculoskeletal],0.726567,-0.726567,statsmodels
72,primary_diag_group[T.Neoplasms]:hba1c_group[T....,0.722161,0.722161,statsmodels



Permutation importance:


,Feature,Importance,Std
76,time_in_hospital,0.065603,0.002451
10,specialty_group[T.Missing or unknown],0.022608,0.001586
7,discharge_group[T.Otherwise],0.015138,0.001527
9,specialty_group[T.Internal Medicine],0.012162,0.001061
80,specialty_group[T.Missing or unknown]:time_in_...,0.011473,0.001050
17,primary_diag_group[T.Musculoskeletal],0.010799,0.001289
21,age_group3[T.Older than 60],0.010546,0.001011
50,specialty_group[T.Other]:age_group3[T.30 years...,0.009578,0.000910
22,age_group3[T.30 years old or younger],0.008207,0.001121
89,primary_diag_group[T.Other]:time_in_hospital,0.007794,0.000958



SHAP importance:


,Feature,MeanAbsSHAP,MeanSHAP
0,time_in_hospital,0.025380,0.000859
1,specialty_group[T.Missing or unknown],0.017387,-0.001096
2,discharge_group[T.Otherwise],0.011743,-0.000500
3,specialty_group[T.Missing or unknown]:time_in_...,0.011397,-0.000301
4,age_group3[T.Older than 60],0.010156,0.001985
5,specialty_group[T.Internal Medicine],0.007391,0.002035
6,primary_diag_group[T.Other]:time_in_hospital,0.007338,0.001248
7,specialty_group[T.Other],0.007266,-0.000365
8,discharge_group[T.Otherwise]:specialty_group[T...,0.007006,-0.000384
9,discharge_group[T.Otherwise]:time_in_hospital,0.006646,0.000542


### Feature coverage


In [25]:
expected_features = len(config["feature_names"])

if native_df is not None:
    print("\nNative features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", expected_features)


expected_native_features = len([
    f for f in config["feature_names"]
    if str(f).lower() not in ["intercept", "const"]
])

if native_df is not None:
    assert len(native_df) == expected_native_features

print("Native features:", len(native_df))
print("Expected native features:", expected_native_features)



Native features: 90
Permutation features: 91
SHAP features: 91
Expected features: 91
Native features: 90
Expected native features: 90


### Numerical integrity


In [26]:
print("\nPermutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

assert perm_df.isna().sum().sum() == 0
assert shap_df.isna().sum().sum() == 0

if native_df is not None:
    assert native_df.isna().sum().sum() == 0

print("NaN validation: OK")



Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0
NaN validation: OK


### Rank agreement


In [27]:
print("\nRank agreement:")
print(global_results["rank_agreement"])

print("\nRank comparison:")
display(global_results["rank_compare"].head(15))



Rank agreement:
{'Native_vs_SHAP': np.float64(0.0474873441165576), 'Native_vs_Permutation': np.float64(0.40850310737951184), 'SHAP_vs_Permutation': np.float64(0.8834259373585217)}

Rank comparison:


,Feature,Importance_x,RawValue,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,specialty_group[T.Surgery]:age_group3[T.30 yea...,2.989244,-2.989244,statsmodels,1.0,0.000123,-0.000123,75.0,0.000674,0.000205,51.0
1,specialty_group[T.Family/general practice]:age...,2.498977,-2.498977,statsmodels,2.0,0.000289,-0.000289,65.0,0.001402,0.000395,44.0
2,specialty_group[T.Other]:age_group3[T.30 years...,2.409895,-2.409895,statsmodels,3.0,0.001741,-0.001741,36.0,0.009578,0.000910,8.0
3,specialty_group[T.Internal Medicine]:age_group...,2.023971,-2.023971,statsmodels,4.0,0.000510,-0.000510,58.0,0.002192,0.000544,31.0
4,age_group3[T.30 years old or younger],1.857647,1.857647,statsmodels,5.0,0.004182,0.004182,18.0,0.008207,0.001121,9.0
5,specialty_group[T.Missing or unknown]:age_grou...,1.471311,-1.471311,statsmodels,6.0,0.001462,-0.001462,39.0,0.004044,0.001098,22.0
6,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.802534,0.802534,statsmodels,7.0,0.000048,0.000048,83.0,0.000222,0.000106,63.0
7,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.796584,0.796584,statsmodels,8.0,0.000104,0.000104,77.0,0.000330,0.000174,56.0
8,primary_diag_group[T.Musculoskeletal],0.726567,-0.726567,statsmodels,9.0,0.006204,-0.000158,11.0,0.010799,0.001289,6.0
9,primary_diag_group[T.Neoplasms]:hba1c_group[T....,0.722161,0.722161,statsmodels,10.0,0.000027,0.000027,86.0,0.000158,0.000083,67.0


### Representative cases


In [28]:
cases = local_results["cases"]

print("\nRepresentative cases:")

if cases is not None and not cases.empty:

    columns_to_show = [
        col for col in [
            "CaseType",
            "y_true",
            "y_pred",
            "y_prob"
        ]
        if col in cases.columns
    ]

    display(cases[columns_to_show])

else:
    print("No representative cases returned.")



Representative cases:


,CaseType,y_true,y_pred,y_prob
48396,TN,0.0,0,0.009532
27989,TN,0.0,0,0.009742
12838,TN,0.0,0,0.010097
20468,TN,0.0,0,0.010383
46540,TN,0.0,0,0.010528
17202,TN,0.0,0,0.011627
27973,TN,0.0,0,0.012171
58480,TN,0.0,0,0.012219
18037,TN,0.0,0,0.012393
23758,TN,0.0,0,0.013231


### Local explanations and seed stability


In [29]:
lime_df = local_results["lime"]
lime_summary_df = local_results["lime_summary"]
lime_seed_stability_df = local_results["lime_seed_stability"]

print("Repeated LIME rows:", len(lime_df))
print("Cases by category:")
display(local_results["cases"].groupby("CaseType").size().rename("N cases"))
print("Seeds per case:")
display(lime_df.groupby(["CaseType", "CaseIndex"])["LIMESeed"].nunique().describe())
print("Aggregated LIME summary:")
display(lime_summary_df.head(20))
print("LIME seed-stability summary:")
display(
    lime_seed_stability_df.groupby("CaseType")[[
        "MeanPairwiseSpearman", "MeanTopKOverlap",
        "MeanRuleWeightSD", "MeanRuleSignConsistency"
    ]].mean().round(3)
)


Repeated LIME rows: 3000
Cases by category:


CaseType
Borderline    10
FN            10
TN            10
Name: N cases, dtype: int64

Seeds per case:


count    30.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max      20.0
Name: LIMESeed, dtype: float64

Aggregated LIME summary:


,CaseType,Rule,MeanWeight,StdWeight,MeanAbsWeight,MedianWeight,PositiveWeightFraction,NExplanations,NCases,NSeeds,SelectionFrequency
0,Borderline,specialty_group[T.Other]:age_group3[T.30 years...,0.464676,0.014705,0.464676,0.464482,1.0,200,10,20,1.000
1,Borderline,specialty_group[T.Internal Medicine]:age_group...,0.411964,0.012278,0.411964,0.411964,1.0,2,1,2,0.010
2,Borderline,specialty_group[T.Missing or unknown]:age_grou...,0.312570,0.016215,0.312570,0.312081,1.0,84,10,20,0.420
3,Borderline,age_group3[T.30 years old or younger] <= 0.00,-0.252629,0.007942,0.252629,-0.252803,0.0,100,5,20,0.500
4,Borderline,age_group3[T.30 years old or younger] > 0.00,0.248366,0.006850,0.248366,0.247415,1.0,100,5,20,0.500
5,Borderline,primary_diag_group[T.Musculoskeletal] <= 0.00,0.152417,0.008595,0.152417,0.152705,1.0,186,10,20,0.930
6,Borderline,time_in_hospital > 6.00,0.147380,0.004137,0.147380,0.147802,1.0,100,5,20,0.500
7,Borderline,specialty_group[T.Surgery]:time_in_hospital <=...,0.129682,0.009796,0.129682,0.129682,1.0,2,2,2,0.010
8,Borderline,specialty_group[T.Internal Medicine] > 0.00,0.094144,NaN,0.094144,0.094144,1.0,1,1,1,0.005
9,Borderline,specialty_group[T.Missing or unknown] <= 0.00,-0.082904,0.003749,0.082904,-0.082790,0.0,180,9,20,0.900


LIME seed-stability summary:


,MeanPairwiseSpearman,MeanTopKOverlap,MeanRuleWeightSD,MeanRuleSignConsistency
CaseType,,,,
Borderline,0.918,0.850,0.032,0.714
FN,0.977,0.941,0.024,0.867
TN,0.938,0.886,0.025,0.818


### Bootstrap-refit stability


In [30]:
if stability_results is None:
    print(
        "\nStability validation: N/A "
        "(expected for HbA1c full-sample statsmodels model)"
    )
else:
    display(stability_results["raw"].head())
    print(stability_results["summary"])



Stability validation: N/A (expected for HbA1c full-sample statsmodels model)


### Exported artefacts


In [31]:
out_dir = Path(config["output_dir"]).resolve()

print("\nOutput directory:")
print(out_dir)

print("\nExported files:")

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))


print("\nHBA1C XAI FRAMEWORK CHECKS COMPLETE (IN-SAMPLE)")



Output directory:
C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\hba1c

Exported files:
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
execution_times.json
lime_explanations.csv
lime_seed_stability.csv
lime_summary.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv

HBA1C XAI FRAMEWORK CHECKS COMPLETE (IN-SAMPLE)


In [32]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)


['Intercept', 'gender[T.Male]', 'race_group[T.Caucasian]', 'race_group[T.Missing]', 'race_group[T.Other]', 'admission_source_group[T.Admitted because of physician/clinic referral]', 'admission_source_group[T.Otherwise]', 'discharge_group[T.Otherwise]', 'specialty_group[T.Family/general practice]', 'specialty_group[T.Internal Medicine]', 'specialty_group[T.Missing or unknown]', 'specialty_group[T.Other]', 'specialty_group[T.Surgery]', 'primary_diag_group[T.Circulatory]', 'primary_diag_group[T.Digestive]', 'primary_diag_group[T.Genitourinary]', 'primary_diag_group[T.Injury]', 'primary_diag_group[T.Musculoskeletal]', 'primary_diag_group[T.Neoplasms]', 'primary_diag_group[T.Other]', 'primary_diag_group[T.Respiratory]', 'age_group3[T.Older than 60]', 'age_group3[T.30 years old or younger]', 'hba1c_group[T.Normal result of the test]', 'hba1c_group[T.Result was high and the diabetic medication was changed]', 'hba1c_group[T.Result was high but the diabetic medication was not changed]', 'discha

,Intercept,gender[T.Male],race_group[T.Caucasian],race_group[T.Missing],race_group[T.Other],admission_source_group[T.Admitted because of physician/clinic referral],admission_source_group[T.Otherwise],discharge_group[T.Otherwise],specialty_group[T.Family/general practice],specialty_group[T.Internal Medicine],...,primary_diag_group[T.Neoplasms]:time_in_hospital,primary_diag_group[T.Other]:time_in_hospital,primary_diag_group[T.Respiratory]:time_in_hospital,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
48396,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.009532,0.490468,TN,48396,1
27989,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,7.0,0.0,0,0.009742,0.490258,TN,27989,2
12838,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.010097,0.489903,TN,12838,3
20468,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.010383,0.489617,TN,20468,4
46540,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,2.0,0.0,0,0.010528,0.489472,TN,46540,5
17202,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.011627,0.488373,TN,17202,6
27973,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.012171,0.487829,TN,27973,7
58480,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,10.0,0.0,0.0,0,0.012219,0.487781,TN,58480,8
18037,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.012393,0.487607,TN,18037,9
23758,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.013231,0.486769,TN,23758,10


In [33]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("HbA1c CaseIndex exists in memory.")


HbA1c CaseIndex exists in memory.


In [34]:
hba1c_cases = pd.read_csv(
    Path("../../results/hba1c/representative_cases.csv")
)

print(hba1c_cases.columns.tolist())
display(hba1c_cases)


['Intercept', 'gender[T.Male]', 'race_group[T.Caucasian]', 'race_group[T.Missing]', 'race_group[T.Other]', 'admission_source_group[T.Admitted because of physician/clinic referral]', 'admission_source_group[T.Otherwise]', 'discharge_group[T.Otherwise]', 'specialty_group[T.Family/general practice]', 'specialty_group[T.Internal Medicine]', 'specialty_group[T.Missing or unknown]', 'specialty_group[T.Other]', 'specialty_group[T.Surgery]', 'primary_diag_group[T.Circulatory]', 'primary_diag_group[T.Digestive]', 'primary_diag_group[T.Genitourinary]', 'primary_diag_group[T.Injury]', 'primary_diag_group[T.Musculoskeletal]', 'primary_diag_group[T.Neoplasms]', 'primary_diag_group[T.Other]', 'primary_diag_group[T.Respiratory]', 'age_group3[T.Older than 60]', 'age_group3[T.30 years old or younger]', 'hba1c_group[T.Normal result of the test]', 'hba1c_group[T.Result was high and the diabetic medication was changed]', 'hba1c_group[T.Result was high but the diabetic medication was not changed]', 'discha

,Intercept,gender[T.Male],race_group[T.Caucasian],race_group[T.Missing],race_group[T.Other],admission_source_group[T.Admitted because of physician/clinic referral],admission_source_group[T.Otherwise],discharge_group[T.Otherwise],specialty_group[T.Family/general practice],specialty_group[T.Internal Medicine],...,primary_diag_group[T.Neoplasms]:time_in_hospital,primary_diag_group[T.Other]:time_in_hospital,primary_diag_group[T.Respiratory]:time_in_hospital,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.009532,0.490468,TN,48396,1
1,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,7.0,0.0,0,0.009742,0.490258,TN,27989,2
2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.010097,0.489903,TN,12838,3
3,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.010383,0.489617,TN,20468,4
4,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,2.0,0.0,0,0.010528,0.489472,TN,46540,5
5,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.011627,0.488373,TN,17202,6
6,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.012171,0.487829,TN,27973,7
7,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,10.0,0.0,0.0,0,0.012219,0.487781,TN,58480,8
8,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.012393,0.487607,TN,18037,9
9,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0,0.013231,0.486769,TN,23758,10


## Interpretation

SHAP and grouped permutation importance were strongly aligned, while absolute coefficient magnitude had almost no rank agreement with SHAP. This difference is expected to be sensitive to scaling, sparse categorical levels, and the interaction-rich Patsy design. All explanations are in-sample. The separate bootstrap sensitivity experiment converged for only 58 of 100 refits and is not an accepted stability result.
